# 10 · Pipeline Tour
**A guided end-to-end walkthrough on a 100-row sample.**

Run every cell top-to-bottom for a complete first run: build → classify → inspect.
No prior setup needed beyond a configured `.env` file.

| Step | What happens |
|------|--------------|
| 1 | Build: reads Excel definition → generates classifiers + taxonomies |
| 2 | Load: samples 100 rows from `Scopus_Content_Collapsed.xlsx` |
| 3 | Classify: runs `contact_reason` + `customer_effort` modules |
| 4 | Inspect: quick look at results vs. ground-truth labels |

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
# ── Project root detection ────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().absolute()
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Project root: {os.getcwd()}")

In [ ]:
import asyncio, json, subprocess
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

# ── Settings — change these to adjust the tour ───────────────────────────
DEFINITION_FILE = "definitions/scopus_content_pipeline_definition_v2.xlsx"
DATA_FILE       = "data/Scopus_Content_Collapsed.xlsx"
OUTPUT_FILE     = "runs/output/tour_sample_enriched.xlsx"
TAXONOMY_DIR    = "generated/taxonomies"

ID_COL          = "Reference #"
TEXT_COL        = "Conversation"
SAMPLE_N        = 100       # rows to classify (set None for full dataset)
SAMPLE_SEED     = 42

MODEL           = "azure/dsaic-llm-general"
BATCH_SIZE      = 10
CONCURRENCY     = 2
RETRIES         = 3
MAX_TOKENS      = 4096

MODULES         = ["contact_reason", "customer_effort"]

## Step 1 · Build
Reads the Excel definition and generates classifier Python files and taxonomy JSONs.
If the generated files are already current, this is a fast no-op.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "build", "--input", DEFINITION_FILE],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

In [ ]:
# Inspect what was generated
manifest_path = Path("generated/build_manifest.json")
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(f"Generator version : {manifest.get('generator_version', '?')}")
    print(f"Generated at      : {manifest.get('generated_at', '?')}")
    print(f"Input file        : {manifest.get('input_file', '?')}")
    print()

config = json.loads(Path("generated/pipeline_config.json").read_text())
for mod, cfg in config["modules"].items():
    from rca_pipeline.taxonomy import load_taxonomy_file
    tax = load_taxonomy_file(Path(TAXONOMY_DIR) / cfg["taxonomy_file"])
    l1  = len(tax)
    l2  = sum(len(n.get("children", {})) for n in tax.values() if isinstance(n, dict))
    print(f"  {mod:<20}  prefix={cfg['taxonomy_prefix']}  L1={l1}  L2 (leaf)={l2}")

## Step 2 · Load data

In [ ]:
df_full = pd.read_excel(DATA_FILE)
print(f"Full dataset: {len(df_full):,} rows  |  columns: {list(df_full.columns)}")

df = df_full.sample(n=min(SAMPLE_N, len(df_full)), random_state=SAMPLE_SEED).reset_index(drop=True)
print(f"Sample      : {len(df):,} rows")
df[[ID_COL, TEXT_COL, "Contact_Reason_Level_1", "Csat Effort Answer Desc"]].head(3)

In [ ]:
# Ground-truth distribution in the sample
print("Contact Reason L1 distribution:")
print(df["Contact_Reason_Level_1"].value_counts().to_string())
print()
print("Effort Answer distribution:")
print(df["Csat Effort Answer Desc"].value_counts().to_string())

## Step 3 · Classify
Runs both modules concurrently and saves a checkpoint after every batch.

In [ ]:
from rca_pipeline.config import RunConfig
from rca_pipeline.taxonomy import load_taxonomies
from rca_pipeline.utils import drop_enrichment_cols, post_process, write_output_file
from rca_pipeline.pipeline import prepare_batched_records, run_classification_module, MODULE_REGISTRY
from rca_pipeline.llm_client import set_cost_tracker_module

# Ensure sample IDs are strings
df[ID_COL] = df[ID_COL].astype(str)

# Write sample to a temp file so read_input_file works
Path("runs/output").mkdir(parents=True, exist_ok=True)
_tmp = "runs/output/_tour_sample_tmp.xlsx"
df.to_excel(_tmp, index=False)

cfg = RunConfig(
    input_file   = _tmp,
    output_file  = OUTPUT_FILE,
    taxonomy_dir = TAXONOMY_DIR,
    id_col       = ID_COL,
    text_col     = TEXT_COL,
    modules      = MODULES,
    model        = MODEL,
    batch_size   = BATCH_SIZE,
    concurrency  = CONCURRENCY,
    retries      = RETRIES,
    max_tokens   = MAX_TOKENS,
)

taxonomies = load_taxonomies(TAXONOMY_DIR)
df_work    = drop_enrichment_cols(df.copy())
batches    = prepare_batched_records(df_work, cfg)
print(f"Batches: {len(batches)}  |  registered modules: {list(MODULE_REGISTRY.keys())}")

In [ ]:
import time

all_results = []

async def _classify():
    for mod in MODULES:
        if mod not in MODULE_REGISTRY:
            print(f"  skipping {mod} — not in registry")
            continue
        t0 = time.time()
        results = await run_classification_module(mod, batches, taxonomies, cfg)
        all_results.extend(results)
        print(f"  {mod:<22}  {len(results)} records  {time.time()-t0:.1f}s")

print("Classifying...")
asyncio.run(_classify())
print(f"Done — {len(all_results)} result records total")

## Step 4 · Inspect results

In [ ]:
# Merge by uid and post-process
grouped = {}
for r in all_results:
    uid = str(r.get("uid", ""))
    grouped.setdefault(uid, {"uid": uid}).update(r)

df_out = post_process(df_work, list(grouped.values()), taxonomies, cfg)
write_output_file(df_out, OUTPUT_FILE)

print(f"Output: {len(df_out)} rows × {len(df_out.columns)} columns → {OUTPUT_FILE}")
enriched_cols = [c for c in df_out.columns if c not in df_work.columns]
print(f"New columns ({len(enriched_cols)}): {enriched_cols}")

In [ ]:
# Quick accuracy check: LLM label vs. ground-truth L1
if "contact_reason_primary_label" in df_out.columns:
    df_cmp = df_out[[ID_COL, "Contact_Reason_Level_1",
                     "contact_reason_primary_code", "contact_reason_primary_label"]].copy()
    print("Pipeline contact_reason_primary_label distribution:")
    print(df_out["contact_reason_primary_label"].value_counts().head(10).to_string())

if "effort_primary_label" in df_out.columns:
    print("\nPipeline effort_primary_label distribution:")
    print(df_out["effort_primary_label"].value_counts().to_string())
    print("\nGround-truth Csat Effort Answer Desc distribution:")
    print(df_out["Csat Effort Answer Desc"].value_counts().to_string())

In [ ]:
# Preview a few rows side-by-side
show_cols = (
    [ID_COL, "Contact_Reason_Level_1", "contact_reason_primary_label"]
    + (["Csat Effort Answer Desc", "effort_score"] if "effort_score" in df_out.columns else [])
)
df_out[[c for c in show_cols if c in df_out.columns]].head(10)